In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class LIFNeuron:
    def __init__(self, tau=20.0, v_rest=0.0, v_reset=0.0, v_thresh=1.0, r=1.0, dt=1.0):
        self.tau = tau
        self.v_rest = v_rest
        self.v_reset = v_reset
        self.v_thresh = v_thresh
        self.r = r
        self.dt = dt
        self.v = v_rest

    def step(self, I):
        dv = (-(self.v - self.v_rest) + self.r * I) / self.tau
        self.v += dv * self.dt
        
        if self.v >= self.v_thresh:
            self.v = self.v_reset
            return 1
        return 0

In [ ]:
neuron = LIFNeuron()
T = 200
I = np.ones(T) * 1.5

v_trace = []
spikes = []

for t in range(T):
    spikes.append(neuron.step(I[t]))
    v_trace.append(neuron.v)

plt.figure(figsize=(10,4))
plt.plot(v_trace)
plt.title("Мембранный потенциал LIF")
plt.show()

plt.figure(figsize=(10,2))
plt.eventplot(np.where(spikes)[0])
plt.title("Raster plot")
plt.show()

In [ ]:
class IzhikevichNeuron:
    def __init__(self, a=0.02, b=0.2, c=-65, d=8, dt=1):
        self.a = a
        self.b = b
        self.c = c
        self.d = d
        self.dt = dt
        self.v = -65
        self.u = self.b * self.v

    def step(self, I):
        dv = 0.04*self.v**2 + 5*self.v + 140 - self.u + I
        du = self.a*(self.b*self.v - self.u)

        self.v += dv*self.dt
        self.u += du*self.dt

        if self.v >= 30:
            self.v = self.c
            self.u += self.d
            return 1
        return 0

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [ ]:
def poisson_encoding(image, time_steps=20):
    return torch.rand(time_steps, *image.shape) < image

In [ ]:
import torch.nn as nn

class SimpleSNN(nn.Module):
    def __init__(self, input_size=784, hidden_size=100, output_size=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size, bias=False)
        self.fc2 = nn.Linear(hidden_size, output_size, bias=False)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.fc2(torch.relu(self.fc1(x)))

In [ ]:
def stdp_update(w, t_pre, t_post, A_plus=0.01, A_minus=0.012):
    delta_t = t_post - t_pre
    if delta_t > 0:
        w += A_plus * np.exp(-delta_t/20)
    else:
        w -= A_minus * np.exp(delta_t/20)
    return np.clip(w, 0, 1)

In [ ]:
model = SimpleSNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(3):
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}, Loss {loss.item()}")

In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in train_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", 100 * correct / total)

In [ ]:
weights = model.fc1.weight.data.numpy()

plt.imshow(weights[0].reshape(28,28))
plt.title("Фильтр первого нейрона")
plt.colorbar()
plt.show()